# Módulo 1 · 02 — Análisis descriptivo e hipótesis

| # | Hipótesis | Cómo se prueba |
|---|---|---|
| H1 | Viernes y domingos tienen más demanda que martes–miércoles | perfil por día de la semana |
| H2 | Festivos y puentes elevan la demanda el día previo y el último día | efecto pre/post festivo |
| H3 | Vacaciones escolares (jun–jul, dic–ene) elevan la demanda | perfil anual |
| H4 | Semana Santa genera un pico | ventana alrededor de Jueves Santo |
| H5 | Fin de año es el pico anual | perfil anual / descomposición |
| H6 | 2020–2021 tiene caída por pandemia y posterior recuperación | tendencia STL |

In [ ]:
# parameters
DEMO = False

In [ ]:
import sys
from pathlib import Path
RAIZ = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(RAIZ))
sys.path.insert(0, str(RAIZ / "modulo1_demanda"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from common.seeds import fijar_semilla, semillas_experimento
from common.config import cargar_config
import m1lib as m1

fijar_semilla()
cfg = cargar_config("modulo1")
M1 = RAIZ / "modulo1_demanda"
DATA, FIG, MOD = M1 / "data", M1 / "figures", M1 / "models"
H, TAU = cfg["pronostico"]["horizonte"], cfg["pronostico"]["cuantil_alto"]
plt.rcParams["figure.figsize"] = (12, 4)

Y = pd.read_parquet(DATA / 'processed' / 'series_diarias.parquet')
cal = pd.read_parquet(DATA / 'processed' / 'calendario.parquet')
rutas = pd.read_csv(DATA / 'processed' / 'rutas.csv', dtype=str)
nombres = dict(zip(rutas.ruta, rutas.nombre))

## Tendencia y estacionalidad (STL semanal y anual — MSTL)

In [ ]:
from statsmodels.tsa.seasonal import MSTL
r0 = rutas.ruta.iloc[0]
res = MSTL(np.log1p(Y[r0]), periods=(7, 365)).fit()
fig = res.plot(); fig.set_size_inches(12, 9)
fig.suptitle(f"Figura 2. Descomposición MSTL (log1p) — {nombres[r0]}"); fig.tight_layout()
fig.savefig(FIG / "fig02_descomposicion.png", dpi=150)

In [ ]:
tendencias = {}
for r in rutas.ruta:
    tendencias[nombres[r]] = MSTL(np.log1p(Y[r]), periods=(7, 365)).fit().trend
pd.DataFrame(tendencias).plot(title="Figura 3. Tendencia (log1p) por ruta — H6 pandemia y recuperación")
plt.axvspan("2020-03-20", "2020-09-01", color="grey", alpha=.2, label="cuarentena")
plt.legend(fontsize=7); plt.savefig(FIG / "fig03_tendencias.png", dpi=150)

## H1 — Día de la semana

In [ ]:
# Demanda relativa: y / media móvil centrada de 7 días (elimina tendencia y nivel de cada ruta)
rel = (Y / Y.rolling(7, center=True).mean()).loc["2022":]
dow = rel.groupby(rel.index.dayofweek).mean()
dow.index = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
dow.rename(columns=nombres).plot(kind="bar", title="Figura 4. Demanda relativa por día de la semana (2022+)")
plt.axhline(1, c="k", lw=.5); plt.savefig(FIG / "fig04_dow.png", dpi=150); dow.round(3)

## H2–H5 — Festivos, puentes, temporadas

In [ ]:
# Demanda relativa a la misma fecha de la semana sin evento (mediana de t-7, t-14, t+7, t+14)
base = pd.concat([Y.shift(k) for k in (7, 14, -7, -14)]).groupby(level=0).median()
efecto = (Y / base).loc["2022":]
c = cal.reindex(efecto.index)
tabla = pd.DataFrame({
    "festivo": efecto[c.festivo == 1].mean(),
    "pre_festivo": efecto[c.pre_festivo == 1].mean(),
    "post_festivo": efecto[c.post_festivo == 1].mean(),
    "puente": efecto[c.puente == 1].mean(),
    "semana_santa": efecto[c.semana_santa == 1].mean(),
    "fin_de_anio": efecto[c.fin_de_anio == 1].mean(),
    "vacaciones_mitad": efecto[c.vacaciones_mitad == 1].mean(),
}).rename(index=nombres)
tabla.round(2)

In [ ]:
anual = (Y / Y.rolling(365, center=True, min_periods=180).mean()).loc["2022":]
perfil = anual.groupby(anual.index.dayofyear).mean().rolling(7, center=True, min_periods=1).mean()
perfil.rename(columns=nombres).plot(title="Figura 5. Perfil anual de demanda relativa (2022+)")
plt.xlabel("día del año"); plt.savefig(FIG / "fig05_perfil_anual.png", dpi=150)

## Conclusiones

_TODO: indicar qué hipótesis se confirmaron (con los números de la tabla) y cuáles no._